# Ticha HTR — cloze correction with local, word-aware attention

This notebook is a variant of **Part 5** of `ticha_trocr_finetuning-modified-transformer.ipynb`. That part trains an ABINet-style cloze model that predicts each character of a line from all the others and uses it to correct the CTC encoder's reading. Its attention has no notion of distance or words: a letter at the other end of the line can count as much as the letter next to it.

Here the cloze attention gets a fixed bias, in the style of ALiBi (Press et al., 2022):

```
score(i, j) = q·k / √d − slope_h · |i − j| − γ · [j is in another word]
            = −∞ when j is in another word and |i − j| > W
```

- **Distance decay per head.** Head *h* has slope 2^−(h+1) (8 heads: 1/2 … 1/256). Steep heads look at the neighbouring letters; flat heads see the whole window.
- **Word penalty γ.** Letters in other words are visible but count less than letters at the same distance in the same word (γ = 1 gives them about 0.37×, γ = 2 about 0.14×).
- **Window W.** Other words are cut off beyond W characters. The current word is always visible (with the distance decay).
- A space belongs to the words on both sides, so predicting a space sees both words.
- Word boundaries come from the line being corrected, so a misread space in the CTC output moves them. Training corrupts spaces too, which teaches some tolerance.

The notebook:

1. Repeats the data cleanup and the text corpora, so the character set matches the saved CTC encoder exactly.
2. Loads the **CTC encoder** saved by Part 2 of the modified-transformer notebook (it is not trained here) and the original cloze models from its Part 5 when they exist (otherwise they are trained here, as the baseline).
3. Trains one local cloze model per configuration in `LOCAL_CONFIGS` and corpus (`zap`, `zap_spa`).
4. Measures **how much attention each model gives to each neighbour**, by offset and same/other word, and draws the attention map for one invented word and one validation line.
5. Corrects the CTC reading with each model (β chosen on validation) and evaluates the test split, next to the test predictions the modified-transformer notebook saved (Parts 3–5), when they exist.

**Run it after the modified-transformer notebook has finished**, at least Part 2 (the encoder) and ideally the whole run, so its test predictions are in the comparison and both don't share the GPU. It reads that notebook's `OUTPUT_DIR` and writes only to `OUTPUT_DIR/cloze_local/`, which the other notebook never reads.

The printed corpus must be in the same place as for the modified-transformer notebook (`data/printed.csv`, `$TICHA_PRINTED_CSV`, or `MyDrive/TichaHTR/printed.csv` in Colab).

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

# The VM's kernel already has these installed; Colab needs them on every new runtime.
if IN_COLAB:
    %pip -q install "transformers==4.46.3" "datasets>=2.19,<4" "accelerate>=0.26" "jiwer>=3.0"

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

import os
# The dataloader workers fork after the tokenizer has run; this avoids a warning per worker.
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import numpy as np
import torch
from datasets import load_dataset
from IPython.display import display

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

ds = load_dataset('maclenn77/ticha-htr-lines', 'original')
print(ds)
for split_name, split_data in ds.items():
    print(f'{split_name}: {len(split_data):,} rows; columns={split_data.column_names}')

In [ ]:
# Audit the existing split by document. Shared doc_id values would make evaluation unreliable.
required_columns = {'image', 'text', 'doc_id'}
for split_name, split_data in ds.items():
    missing = required_columns - set(split_data.column_names)
    if missing:
        raise ValueError(f'{split_name} is missing required columns: {missing}')

doc_sets = {
    name: {str(x) for x in data['doc_id'] if x is not None}
    for name, data in ds.items()
}
overlaps = {
    'train-validation': doc_sets['train'] & doc_sets['validation'],
    'train-test': doc_sets['train'] & doc_sets['test'],
    'validation-test': doc_sets['validation'] & doc_sets['test'],
}
for pair, shared_docs in overlaps.items():
    print(f'{pair} shared document IDs: {len(shared_docs)}')
    if shared_docs:
        print('Examples:', list(sorted(shared_docs))[:10])

for split_name, split_data in ds.items():
    missing_doc_ids = sum(x is None for x in split_data['doc_id'])
    print(f'{split_name}: rows without doc_id = {missing_doc_ids}')

if any(overlaps.values()):
    print('WARNING: document leakage detected. Do not train until document-level splits are fixed.')
else:
    print('Good: no shared non-null document IDs were found across splits.')

## Clean transcriber notes

The transcriptions mark expansions, glosses and damage with `[]` and `()`. This cell turns them into diplomatic text (what is written on the page) and drops lines whose image and text don't match. Uncertain readings such as `(Santigo?)` are kept as text in train and dropped from validation and test, so the reported CER is on confident transcriptions only (state this in the report). It changes the text only for this experiment: the Hub dataset stays as published, and the original text is kept in `text_raw`.

Copied unchanged from the modified-transformer notebook: the CTC character set is built from this cleaned text, so any change here breaks the saved encoder.

In [ ]:
# Transcriber notes: make the text match what is written on the page (diplomatic transcription).
# The transcriptions use [] and () for three things:
#   - abbreviation expansions, p[es]os, to[mine]s, Dom(ing)o: the bracketed letters are not written,
#     so they are deleted (p[es]os -> pos)
#   - glosses, [=Juan]: not on the page, deleted
#   - damage, illegible text and editorial notes, [cortado], [roto], [xxx], [...], [?], [n?],
#     [margin note], (?), (word?): the image has ink the text doesn't cover, so the whole line is dropped
#   - uncertain readings, (Santigo?): the letters are on the page but the transcriber wasn't sure.
#     In train they are kept as text (Santigo); in validation and test the line is dropped, so
#     evaluation uses only confident transcriptions.
# Lines with unbalanced or nested brackets are dropped too.
import re

BRACKET_RE = re.compile(r'\[([^\[\]]*)\]|\(([^()]*)\)')
NOTE_RE = re.compile(
    r'[?…]|^\.+$|\.\.|\s|,|^x+$|^word$'                   # uncertain, illegible, or a note with spaces
    r'|cortado|roto|torn|mancha|borrado|firmas|ilegible',  # damage and editorial notes
    re.IGNORECASE)
DROP_HASH = True  # '#' marks illegible characters in some documents
KEEP_GUESSES_IN_TRAIN = True
GUESS_RE = re.compile(r'\((?!word\?)([^()?]*[^\W\d_][^()?]*)\?+\)')  # (letters?) but not (word?)


def clean_text(text, keep_guesses=False):
    """Return the diplomatic text, or None when the line should be dropped."""
    text = str(text or '')
    if keep_guesses:
        text = GUESS_RE.sub(lambda m: m.group(1), text)
    notes = [square or round for square, round in BRACKET_RE.findall(text)]
    if any(NOTE_RE.search(note.strip()) for note in notes if not note.strip().startswith('=')):
        return None
    text = BRACKET_RE.sub('', text)
    if re.search(r'[\[\]()]', text) or (DROP_HASH and '#' in text):
        return None
    return re.sub(r'\s+', ' ', text).strip() or None


def clean_batch(texts, keep_guesses):
    return {'text_raw': texts, 'text': [clean_text(t, keep_guesses) for t in texts]}

for split_name in ds:
    before = len(ds[split_name])
    keep_guesses = KEEP_GUESSES_IN_TRAIN and split_name == 'train'
    # input_columns keeps the images from being decoded.
    ds[split_name] = ds[split_name].map(clean_batch, input_columns='text', batched=True,
                                        fn_kwargs={'keep_guesses': keep_guesses})
    ds[split_name] = ds[split_name].filter(lambda t: t is not None, input_columns='text')
    changed = sum(raw != text for raw, text in zip(ds[split_name]['text_raw'], ds[split_name]['text']))
    print(f'{split_name}: kept {len(ds[split_name]):,} / {before:,} lines '
          f'(dropped {before - len(ds[split_name]):,}, cleaned text in {changed:,})')

examples = [(r, t) for r, t in zip(ds['train']['text_raw'], ds['train']['text']) if r != t][:5]
for raw, text in examples:
    print(f'{raw!r}\n  -> {text!r}')

## Remove train documents that duplicate evaluation documents

Ticha lists some manuscripts twice. **Th687 (train) and Za687 (test) are the same pages**, scanned twice and transcribed the same way, so without this step the test split would score lines the model was trained on. A train document is dropped when more than half of its lines appear verbatim in validation or test.

Original/translation pairs (`Al710` and `Al710T`) are different manuscripts in different languages, but they share names, places and dates. They are kept by default (about 10% of the train lines); set `DROP_TRANSLATION_PAIRS = True` to drop the train half of each pair and report both numbers if the difference matters.

In [ ]:
DUPLICATE_SHARE = 0.5
DROP_TRANSLATION_PAIRS = False


def line_key(text):
    return re.sub(r'\W+', ' ', text.lower()).strip()


def base_id(doc_id):
    return doc_id[:-1] if doc_id.endswith('T') else doc_id


eval_keys = {line_key(text) for split_name in ('validation', 'test')
             for text in ds[split_name]['text'] if len(text) >= 15}
train_lines_by_doc = {}
for doc_id, text in zip(ds['train']['doc_id'], ds['train']['text']):
    train_lines_by_doc.setdefault(doc_id, []).append(text)
duplicate_docs = {doc_id for doc_id, texts in train_lines_by_doc.items()
                  if sum(line_key(t) in eval_keys for t in texts) > DUPLICATE_SHARE * len(texts)}
eval_bases = {base_id(d) for split_name in ('validation', 'test') for d in set(ds[split_name]['doc_id'])}
pair_docs = {doc_id for doc_id in train_lines_by_doc if base_id(doc_id) in eval_bases}

print('Train documents duplicating validation/test documents (dropped):', sorted(duplicate_docs))
print(f"Train documents paired with a validation/test translation: {len(pair_docs)} "
      f"({sum(len(train_lines_by_doc[d]) for d in pair_docs)} lines), "
      f"{'dropped' if DROP_TRANSLATION_PAIRS else 'kept'}")
drop_docs = duplicate_docs | (pair_docs if DROP_TRANSLATION_PAIRS else set())
before = len(ds['train'])
ds['train'] = ds['train'].filter(lambda doc_id: doc_id not in drop_docs, input_columns='doc_id')
# Za687 (test) is the same manuscript as Th687 (train in dataset v1.0), so it is left out of the test split:
# every model is tested on the same lines, none of which a model has seen in training.
TEST_EXCLUDE = {'Za687'}
ds['test'] = ds['test'].filter(lambda doc_id: doc_id not in TEST_EXCLUDE, input_columns='doc_id')
print(f"test: {len(ds['test']):,} lines without {', '.join(sorted(TEST_EXCLUDE))}")
print(f"train: kept {len(ds['train']):,} / {before:,} lines")

## Setup

Same output folder as the modified-transformer notebook, to read its saved models. New files go to `OUTPUT_DIR/cloze_local/`.

In [ ]:
import json
import math
import re
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn.functional as F
from collections import Counter
from jiwer import cer, wer
from torch.utils.data import DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_linear_schedule_with_warmup

if any(overlaps.values()):
    raise RuntimeError('Document IDs overlap across splits. Fix document-level splits before training.')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab choose Runtime > Change runtime type > T4 GPU; on the VM check nvidia-smi. Then rerun cells.')

MODEL_ID = 'microsoft/trocr-base-handwritten'
processor = TrOCRProcessor.from_pretrained(MODEL_ID)

if IN_COLAB:
    OUTPUT_DIR = '/content/drive/MyDrive/TichaHTR/trocr-modified'
else:
    OUTPUT_DIR = os.path.expanduser(os.environ.get('TICHA_OUTPUT_DIR', '~/ticha-runs/trocr-modified'))
LOCAL_DIR = os.path.join(OUTPUT_DIR, 'cloze_local')
os.makedirs(LOCAL_DIR, exist_ok=True)
# Every model is saved and, when its file already exists, loaded instead of trained. Set RETRAIN = True to train
# this notebook's models again (the encoder and the modified-transformer notebook's models are never retrained here).
RETRAIN = False
curves = {}

USE_BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
DEVICE = torch.device('cuda')
NUM_WORKERS = 2 if IN_COLAB else 4
print('Reading from:', OUTPUT_DIR)
print('Writing to:', LOCAL_DIR)

## Text corpora

Copied from Part 1 of the modified-transformer notebook (without the per-work sample printout). The cloze models train on `text_corpora` and are validated on `lm_eval_texts`.

In [ ]:
import math
import re
import unicodedata
from collections import Counter
import pandas as pd

if IN_COLAB:
    PRINTED_CSV = '/content/drive/MyDrive/TichaHTR/printed.csv'
else:
    # The kernel may start in the repo root (VS Code, the VM's Jupyter service) or in notebooks/train/.
    PRINTED_CSV = os.environ.get('TICHA_PRINTED_CSV') or next(
        (p for p in ['data/printed.csv', '../../data/printed.csv'] if os.path.exists(p)), 'data/printed.csv')
    PRINTED_CSV = os.path.abspath(os.path.expanduser(PRINTED_CSV))
if not os.path.exists(PRINTED_CSV):
    raise FileNotFoundError(f'Printed corpus not found at {PRINTED_CSV}; set TICHA_PRINTED_CSV (cwd is {os.getcwd()}).')
print('Printed corpus:', PRINTED_CSV)
MANUSCRIPT_REPEATS = 2

# Printed-only fixes. The manuscripts are transcribed with plain s, so long s is normalized.
PRINTED_FIXES = str.maketrans({'ſ': 's', 'ҫ': 'ç', 'С': 'Ç',
                               '¶': None, '|': None, '¦': None, '*': None, '✠': None, '℞': None})


MANUSCRIPT_CHARS = set(''.join(ds['train']['text']))


def to_manuscript_chars(line):
    """Strip accents the manuscripts don't use; return None if a character still isn't in the manuscripts."""
    chars = []
    for ch in line:
        if ch not in MANUSCRIPT_CHARS:
            ch = ''.join(c for c in unicodedata.normalize('NFD', ch) if unicodedata.category(c) != 'Mn')
            if not ch or any(c not in MANUSCRIPT_CHARS for c in ch):
                return None
        chars.append(ch)
    return ''.join(chars)


def clean_printed_line(line):
    """Return a printed line in the manuscripts' conventions, or None when it should be dropped."""
    line = unicodedata.normalize('NFC', line).translate(PRINTED_FIXES)
    line = ''.join(ch for ch in line if unicodedata.category(ch) != 'Co')  # private-use glyphs
    line = re.sub(r'\s+', ' ', line).strip()
    if any(ch.isdigit() for ch in line) and not any(ch.islower() for ch in line):
        return None  # running heads ('ZAPOTECA 4'), page numbers, printer's signatures
    if len(re.findall(r'[^\W\d_]', line)) < 3:
        return None
    if re.fullmatch(r'(?:\S ){2,}\S[.,]?', line):
        return None  # letter-spaced title lines: 'A R T E E N L E N'
    line = to_manuscript_chars(clean_text(line, keep_guesses=True) or '')
    return line or None


WORD_RE = re.compile(r'[^\W\d_]+')
# A printed line is Zapotec when at least this share of its letters is in Zapotec words. 0.7 keeps Zapotec lines
# with Spanish loanwords (Dios, Santa Maria, gracia) and still lets no Spanish validation line through.
ZAPOTEC_SHARE = 0.7


class WordLanguageModel:
    """Character n-gram naive Bayes over word types: whether a word is Zapotec or Spanish."""

    def __init__(self, words_by_language, n=4, alpha=0.5):
        self.n, self.alpha = n, alpha
        self.counts = {language: Counter(g for w in words for g in self.grams(w))
                       for language, words in words_by_language.items()}
        self.totals = {language: sum(c.values()) for language, c in self.counts.items()}
        self.vocab_size = len(set().union(*self.counts.values()))
        n_words = sum(len(words) for words in words_by_language.values())
        self.prior = {language: math.log(len(words) / n_words) for language, words in words_by_language.items()}

    def grams(self, word):
        word = f'<{word}>'
        return [word[i:i + k] for k in range(1, self.n + 1) for i in range(len(word) - k + 1)]

    def log_score(self, word, language):
        counts, total = self.counts[language], self.totals[language] + self.alpha * self.vocab_size
        return self.prior[language] + sum(math.log((counts[g] + self.alpha) / total) for g in self.grams(word))

    def is_zapotec(self, word):
        return self.log_score(word, 'zap') > self.log_score(word, 'spa')


def words_of(text):
    text = unicodedata.normalize('NFC', text.lower()).translate(PRINTED_FIXES)
    return [w for w in WORD_RE.findall(text) if len(w) >= 2]


printed = pd.read_csv(PRINTED_CSV)

# Labelled words: Ticha marks every Zapotec word in Córdova's grammar, so the rest of his text is Spanish.
# Manuscript train lines add the documents' spelling; words also seen in Spanish documents (names, dates,
# loanwords) are not counted as Zapotec.
zap_words, spa_words = Counter(), Counter()
for page, marked in printed.loc[printed['zapotec_marked'].notna(), ['transcription', 'zapotec_marked']].values:
    marked_words = set(words_of(marked))
    for word in words_of(page):
        (zap_words if word in marked_words else spa_words)[word] += 1
for text, language in zip(ds['train']['text'], ds['train']['language']):
    if language == 'Spanish':
        spa_words.update(words_of(text))
for text, language in zip(ds['train']['text'], ds['train']['language']):
    if language == 'Zapotec':
        zap_words.update(w for w in words_of(text) if w not in spa_words)
word_model = WordLanguageModel({'zap': list(zap_words), 'spa': list(spa_words)})


def is_zapotec_line(line):
    words = words_of(line)
    if len(words) < 2:
        return False
    zapotec_letters = sum(len(w) for w in words if word_model.is_zapotec(w))
    return zapotec_letters >= ZAPOTEC_SHARE * sum(map(len, words))


printed_rows = [(slug, cleaned)
                for slug, page in printed[['slug', 'transcription']].dropna().values
                for line in page.split('\n')
                if (cleaned := clean_printed_line(line))]
seen_lines = set()
printed_rows = [(slug, line) for slug, line in printed_rows if not (line in seen_lines or seen_lines.add(line))]
printed_lines = [line for _, line in printed_rows]
printed_zap = [line for line in printed_lines if is_zapotec_line(line)]

train_pairs = list(zip(ds['train']['text'], ds['train']['language']))
manuscript_zap = [text for text, language in train_pairs if language == 'Zapotec']
manuscript_all = [text for text, _ in train_pairs]

text_corpora = {
    'zap': manuscript_zap * MANUSCRIPT_REPEATS + printed_zap,
    'zap_spa': manuscript_all * MANUSCRIPT_REPEATS + printed_lines,
}
# Validation lines measure how well each decoder predicts unseen manuscript text, per language.
val_pairs = list(zip(ds['validation']['text'], ds['validation']['language']))
lm_eval_texts = {
    'zap': [text for text, language in val_pairs if language == 'Zapotec'],
    'spa': [text for text, language in val_pairs if language != 'Zapotec'],
}

print(f'Printed: {len(printed_lines):,} lines kept, {len(printed_zap):,} classified as Zapotec')
print(f'Manuscript train: {len(manuscript_all):,} lines, {len(manuscript_zap):,} from Zapotec documents')
for name, texts in text_corpora.items():
    print(f'Corpus {name}: {len(texts):,} lines, {sum(map(len, texts)):,} characters')
for name, texts in lm_eval_texts.items():
    print(f'LM validation {name}: {len(texts):,} lines')

## Load the CTC encoder (Part 2 of the modified-transformer notebook)

The encoder and its CTC head are loaded, not trained. The assertion checks that the cleanup above produced the same character set the encoder was trained with.

In [ ]:
CTC_BATCH_SIZE = 16
ENCODER_PATH = os.path.join(OUTPUT_DIR, 'encoder_ctc', 'encoder_ctc.pt')
if not os.path.exists(ENCODER_PATH):
    raise FileNotFoundError(f'{ENCODER_PATH} not found: run Part 2 of ticha_trocr_finetuning-modified-transformer.ipynb first.')
saved_encoder = torch.load(ENCODER_PATH, map_location='cpu')

ctc_chars = sorted(set(''.join(ds['train']['text'])))
assert saved_encoder['chars'] == ctc_chars, 'The cleanup differs from the one the encoder was trained with.'
char_to_id = {ch: i + 1 for i, ch in enumerate(ctc_chars)}  # 0 is the CTC blank
CTC_UPSAMPLE = saved_encoder['upsample']
best_cer = saved_encoder['best_cer']


class EncoderCTC(torch.nn.Module):
    """TrOCR's ViT encoder with a column-wise CTC head."""

    def __init__(self, encoder, n_classes, upsample):
        super().__init__()
        self.encoder = encoder
        self.grid = encoder.config.image_size // encoder.config.patch_size
        self.upsample = upsample
        self.n_classes = n_classes
        hidden = encoder.config.hidden_size
        self.column_proj = torch.nn.Sequential(
            torch.nn.Linear(self.grid * hidden, 512), torch.nn.GELU(), torch.nn.Dropout(0.1))
        self.head = torch.nn.Linear(512, upsample * n_classes)

    def forward(self, pixel_values):
        patches = self.encoder(pixel_values=pixel_values).last_hidden_state[:, -self.grid ** 2:]  # drop CLS
        b = patches.size(0)
        columns = patches.reshape(b, self.grid, self.grid, -1).permute(0, 2, 1, 3).reshape(b, self.grid, -1)
        logits = self.head(self.column_proj(columns))
        return logits.reshape(b, self.grid * self.upsample, self.n_classes)


class CTCCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        images = [item['image'].convert('RGB') for item in batch]
        pixel_values = self.processor(images=images, return_tensors='pt').pixel_values
        texts = [str(item['text'] or '').strip() for item in batch]
        ids = [[char_to_id[ch] for ch in text if ch in char_to_id] for text in texts]
        return {
            'pixel_values': pixel_values,
            'targets': torch.tensor([i for seq in ids for i in seq], dtype=torch.long),
            'target_lengths': torch.tensor([len(seq) for seq in ids], dtype=torch.long),
            'texts': texts,
        }


def ctc_decode(logits):
    """Greedy CTC: best class per slot, merge repeats, drop blanks."""
    decoded = []
    for seq in logits.argmax(-1).cpu().tolist():
        chars, prev = [], 0
        for i in seq:
            if i != prev and i != 0:
                chars.append(ctc_chars[i - 1])
            prev = i
        decoded.append(re.sub(r'\s+', ' ', ''.join(chars)).strip())
    return decoded


@torch.no_grad()
def ctc_predict(ctc_model, split):
    loader = DataLoader(split, batch_size=CTC_BATCH_SIZE, collate_fn=ctc_collator, num_workers=NUM_WORKERS)
    ctc_model.eval()
    predictions, references = [], []
    for batch in loader:
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            logits = ctc_model(batch['pixel_values'].to(DEVICE))
        predictions += ctc_decode(logits.float())
        references += batch['texts']
    return predictions, references


ctc_collator = CTCCollator(processor)
ctc_model = EncoderCTC(VisionEncoderDecoderModel.from_pretrained(MODEL_ID).encoder,
                       len(ctc_chars) + 1, CTC_UPSAMPLE)
ctc_model.load_state_dict(saved_encoder['ctc_model'])
ctc_model = ctc_model.to(DEVICE).eval()
del saved_encoder
print(f'Loaded the CTC encoder: {len(ctc_chars)} characters, validation CER {best_cer:.3f}')

## Cloze models: the original one and the local, word-aware one

`ClozeCharLM` is the Part 5 model, with the same parameters, so the modified-transformer notebook's saved models load into it. Its attention mask is now built in `attention_masks`, and `forward(..., return_weights=True)` also returns the attention weights for the analysis below.

`LocalClozeCharLM` replaces that mask with the bias described at the top. It adds no trainable parameters, only the fixed slopes, `W` and `γ`.

Each entry of `LOCAL_CONFIGS` is one model per corpus. Training is short (a small 4-layer model on text), so trying several is cheap; add entries to compare more windows and penalties. The configuration is chosen by validation CER of the correction, never by test.

In [ ]:
CLOZE_VARIANTS = ['zap', 'zap_spa']
CLOZE_DIM, CLOZE_LAYERS, CLOZE_HEADS = 256, 4, 8
CLOZE_EPOCHS = 10
CLOZE_BATCH_SIZE = 128
CLOZE_LR = 5e-4
CLOZE_NOISE = min(0.3, max(0.05, best_cer))  # as in Part 5: context noise close to the CTC encoder's CER
CLOZE_ITERATIONS = 2
CLOZE_BETAS = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
CLOZE_MAX_LENGTH = ctc_model.grid * CTC_UPSAMPLE
n_ctc_classes = len(ctc_chars) + 1
SPACE_ID = char_to_id[' ']

# window: characters of another word further away than this are not visible.
# word_penalty: logit penalty on characters of another word (exp(-1) = 0.37x, exp(-2) = 0.14x).
LOCAL_CONFIGS = {
    'near': {'window': 8, 'word_penalty': 2.0},
    'wide': {'window': 16, 'word_penalty': 1.0},
}


class ClozeLayer(torch.nn.Module):
    """Cross-attention from the position queries to the characters, then a feed-forward block."""

    def __init__(self, dim, heads, dropout=0.1):
        super().__init__()
        self.norm_query = torch.nn.LayerNorm(dim)
        self.attn = torch.nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm_ff = torch.nn.LayerNorm(dim)
        self.ff = torch.nn.Sequential(torch.nn.Linear(dim, 4 * dim), torch.nn.GELU(),
                                      torch.nn.Dropout(dropout), torch.nn.Linear(4 * dim, dim))
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, query, memory, attn_mask, padding_mask, need_weights=False):
        q = self.norm_query(query)
        attended, weights = self.attn(q, memory, memory, attn_mask=attn_mask, key_padding_mask=padding_mask,
                                      need_weights=need_weights, average_attn_weights=False)
        query = query + self.dropout(attended)
        return query + self.dropout(self.ff(self.norm_ff(query))), weights


class ClozeCharLM(torch.nn.Module):
    """Predicts every character of a line from the other characters (ABINet-style cloze model, Part 5)."""

    def __init__(self, n_classes, max_length, dim, layers, heads):
        super().__init__()
        self.heads = heads
        self.char_emb = torch.nn.Embedding(n_classes, dim, padding_idx=0)
        self.pos_emb = torch.nn.Embedding(max_length, dim)
        self.null = torch.nn.Parameter(torch.zeros(1, 1, dim))
        self.memory_norm = torch.nn.LayerNorm(dim)
        self.layers = torch.nn.ModuleList(ClozeLayer(dim, heads) for _ in range(layers))
        self.norm = torch.nn.LayerNorm(dim)
        self.head = torch.nn.Linear(dim, n_classes)

    def attention_masks(self, ids):
        """Position i can't see character i; the null slot (column 0) is always visible."""
        b, n = ids.shape
        attn_mask = torch.zeros(n, n + 1, dtype=torch.bool, device=ids.device)
        attn_mask[:, 1:] = torch.eye(n, dtype=torch.bool, device=ids.device)
        padding_mask = torch.cat([torch.zeros(b, 1, dtype=torch.bool, device=ids.device), ids == 0], 1)
        return attn_mask, padding_mask

    def forward(self, ids, return_weights=False):
        b, n = ids.shape
        positions = self.pos_emb(torch.arange(n, device=ids.device))
        memory = self.memory_norm(self.char_emb(ids) + positions)
        memory = torch.cat([self.null.expand(b, -1, -1), memory], 1)
        attn_mask, padding_mask = self.attention_masks(ids)
        query = positions.unsqueeze(0).expand(b, -1, -1)
        weights = []
        for layer in self.layers:
            query, layer_weights = layer(query, memory, attn_mask, padding_mask, need_weights=return_weights)
            weights.append(layer_weights)
        logits = self.head(self.norm(query))
        return (logits, weights) if return_weights else logits


def word_spans(ids):
    """First and last word index of each character. A space belongs to the words on both sides."""
    spaces = (ids == SPACE_ID).long()
    last = spaces.cumsum(-1)
    return last - spaces, last


def same_word(ids):
    """(batch, n, n): whether characters i and j share a word."""
    first, last = word_spans(ids)
    return (first[:, :, None] <= last[:, None, :]) & (first[:, None, :] <= last[:, :, None])


class LocalClozeCharLM(ClozeCharLM):
    """The cloze model with an ALiBi-style distance decay per head, a penalty on other words and a window."""

    def __init__(self, n_classes, max_length, dim, layers, heads, window, word_penalty):
        super().__init__(n_classes, max_length, dim, layers, heads)
        self.window, self.word_penalty = window, word_penalty
        slopes = torch.tensor([2.0 ** -(h + 1) for h in range(heads)])
        self.register_buffer('slopes', slopes, persistent=False)

    def attention_masks(self, ids):
        b, n = ids.shape
        pos = torch.arange(n, device=ids.device)
        dist = (pos[:, None] - pos[None, :]).abs()
        other = ~same_word(ids)
        bias = (-self.slopes.view(1, -1, 1, 1) * dist.float()
                - self.word_penalty * other[:, None].float())                       # (b, heads, n, n)
        blocked = ((other & (dist > self.window))[:, None]                          # other words beyond W
                   | torch.eye(n, dtype=torch.bool, device=ids.device)              # its own character
                   | (ids == 0)[:, None, None, :])                                  # padding
        bias = bias.masked_fill(blocked, float('-inf'))
        null = torch.zeros(b, self.heads, n, 1, device=ids.device)                 # always visible
        return torch.cat([null, bias], -1).reshape(b * self.heads, n, n + 1), None


def cloze_sequences(texts):
    seqs = [[char_to_id[ch] for ch in text if ch in char_to_id][:CLOZE_MAX_LENGTH] for text in texts]
    return [seq for seq in seqs if seq]


def pad_ids(seqs):
    ids = torch.zeros(len(seqs), max(max(map(len, seqs)), 1), dtype=torch.long)
    for row, seq in enumerate(seqs):
        ids[row, :len(seq)] = torch.tensor(seq)
    return ids


def add_context_noise(ids):
    noisy = (torch.rand(ids.shape, device=ids.device) < CLOZE_NOISE) & (ids != 0)
    return torch.where(noisy, torch.randint(1, n_ctc_classes, ids.shape, device=ids.device), ids)


@torch.no_grad()
def cloze_accuracy(cloze_lm, seqs):
    cloze_lm.eval()
    correct = total = 0
    for start in range(0, len(seqs), CLOZE_BATCH_SIZE):
        ids = pad_ids(seqs[start:start + CLOZE_BATCH_SIZE]).to(DEVICE)
        predicted = cloze_lm(ids).argmax(-1)
        correct += ((predicted == ids) & (ids != 0)).sum().item()
        total += (ids != 0).sum().item()
    return correct / max(total, 1)


def train_cloze(cloze_lm, variant, path, name):
    """Part 5's training loop: noisy context, cross-entropy on every character, best validation epoch kept."""
    torch.manual_seed(42)
    train_seqs = cloze_sequences(text_corpora[variant])
    best_accuracy, best_state, history = -1.0, None, []
    optimizer = torch.optim.AdamW(cloze_lm.parameters(), lr=CLOZE_LR, weight_decay=0.01)
    steps_per_epoch = math.ceil(len(train_seqs) / CLOZE_BATCH_SIZE)
    scheduler = get_linear_schedule_with_warmup(optimizer, steps_per_epoch // 2, CLOZE_EPOCHS * steps_per_epoch)
    for epoch in range(1, CLOZE_EPOCHS + 1):
        cloze_lm.train()
        order = torch.randperm(len(train_seqs)).tolist()
        running_loss = 0.0
        for start in range(0, len(order), CLOZE_BATCH_SIZE):
            ids = pad_ids([train_seqs[i] for i in order[start:start + CLOZE_BATCH_SIZE]]).to(DEVICE)
            logits = cloze_lm(add_context_noise(ids))
            loss = F.cross_entropy(logits.reshape(-1, n_ctc_classes), ids.reshape(-1), ignore_index=0)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(cloze_lm.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()
        accuracy = {split: cloze_accuracy(cloze_lm, seqs) for split, seqs in cloze_eval_seqs.items()}
        print(f'{name} epoch {epoch}: loss {running_loss / steps_per_epoch:.3f} | validation accuracy '
              + ', '.join(f'{split} {acc:.3f}' for split, acc in accuracy.items()))
        history.append({'epoch': epoch, 'train_loss': running_loss / steps_per_epoch,
                        **{f'eval_{split}_accuracy': acc for split, acc in accuracy.items()}})
        if np.mean(list(accuracy.values())) > best_accuracy:
            best_accuracy, best_accuracies = np.mean(list(accuracy.values())), accuracy
            best_state = {k: v.detach().cpu().clone() for k, v in cloze_lm.state_dict().items()}
    cloze_lm.load_state_dict(best_state)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({'state': cloze_lm.state_dict(), 'chars': ctc_chars, 'accuracy': best_accuracies,
                'history': history}, path)
    return best_accuracies, history


cloze_eval_seqs = {split: cloze_sequences(texts) for split, texts in lm_eval_texts.items()}
print(f'Context noise {CLOZE_NOISE:.3f} | word-aware configurations: {list(LOCAL_CONFIGS)}')

## Train (or load) the models

For each corpus: the original cloze model (loaded from the modified-transformer notebook's `cloze_<variant>/` when it exists, otherwise trained here into `cloze_local/`), then one local model per configuration.

In [ ]:
def new_model(config=None):
    if config is None:
        return ClozeCharLM(n_ctc_classes, CLOZE_MAX_LENGTH, CLOZE_DIM, CLOZE_LAYERS, CLOZE_HEADS).to(DEVICE)
    return LocalClozeCharLM(n_ctc_classes, CLOZE_MAX_LENGTH, CLOZE_DIM, CLOZE_LAYERS, CLOZE_HEADS,
                            **config).to(DEVICE)


cloze_models, cloze_results = {}, {}
for variant in CLOZE_VARIANTS:
    runs = [(f'original {variant}', None, os.path.join(OUTPUT_DIR, f'cloze_{variant}', 'cloze_lm.pt'),
             os.path.join(LOCAL_DIR, f'cloze_{variant}', 'cloze_lm.pt'))]
    runs += [(f'local {config_name} {variant}', config, None,
              os.path.join(LOCAL_DIR, f'local_{config_name}_{variant}', 'cloze_lm.pt'))
             for config_name, config in LOCAL_CONFIGS.items()]
    for name, config, part5_path, path in runs:
        cloze_lm = new_model(config)
        saved_path = next((p for p in [part5_path, path] if p and os.path.exists(p)), None)
        if saved_path and not (RETRAIN and saved_path == path):
            saved = torch.load(saved_path, map_location='cpu')
            assert saved['chars'] == ctc_chars, f'{saved_path} used another character set.'
            cloze_lm.load_state_dict(saved['state'])
            cloze_results[name], curves[name] = saved['accuracy'], saved['history']
            print(f'{name}: using the saved model {saved_path}')
        else:
            cloze_results[name], curves[name] = train_cloze(cloze_lm, variant, path, name)
        cloze_models[name] = cloze_lm.eval()

# Share of validation manuscript characters predicted from their context alone.
display(pd.DataFrame(cloze_results).T.round(3))

### Learning curves

Training loss and validation accuracy per epoch. The best validation epoch is the one kept; a falling loss with flat or falling accuracy is overfitting.

In [ ]:
fig, axes = plt.subplots(len(curves), 1, figsize=(8, 2.4 * len(curves)), squeeze=False)
for ax, (name, history) in zip(axes[:, 0], curves.items()):
    frame = pd.DataFrame(history).set_index('epoch')
    frame[['train_loss']].plot(ax=ax, marker='o')
    frame[[c for c in frame if c.endswith('accuracy')]].plot(ax=ax.twinx(), marker='s', linestyle='--')
    ax.set_title(name)
plt.tight_layout()
plt.show()

## Where does the attention go?

For every validation character, the attention each model gives to the other characters, averaged over heads and layers, grouped by offset (key position − query position) and by whether the key is in the same word. The value is the mean weight **per character** at that offset, so it answers "does the `z` next to the `i` count more than the `d` two places away?".

`null` is the share that goes to the null slot (the model's way of attending to nothing). In the original model nothing ties the weights to distance, so any locality it shows was learned; in the local models it is imposed.

In [ ]:
PROFILE_VARIANT = 'zap'
MAX_OFFSET = 8


@torch.no_grad()
def attention_profile(cloze_lm, seqs):
    cloze_lm.eval()
    sums, counts, shares, n_queries = Counter(), Counter(), Counter(), 0
    for start in range(0, len(seqs), CLOZE_BATCH_SIZE):
        ids = pad_ids(seqs[start:start + CLOZE_BATCH_SIZE]).to(DEVICE)
        _, weights = cloze_lm(ids, return_weights=True)
        w = torch.stack(weights).float().mean((0, 2))          # layers and heads -> (b, n, n + 1)
        null, w = w[..., 0], w[..., 1:]
        b, n = ids.shape
        pos = torch.arange(n, device=DEVICE)
        offset = (pos[None, :] - pos[:, None]).expand(b, n, n)
        same = same_word(ids)
        real = (ids != 0)[:, :, None] & (ids != 0)[:, None, :] & (offset != 0)
        clipped = offset.clamp(-MAX_OFFSET - 1, MAX_OFFSET + 1)
        for o in range(-MAX_OFFSET - 1, MAX_OFFSET + 2):
            for is_same in (True, False):
                selected = real & (clipped == o) & (same == is_same)
                sums[o, is_same] += w[selected].sum().item()
                counts[o, is_same] += selected.sum().item()
        shares['same word'] += w[real & same].sum().item()
        shares['other words'] += w[real & ~same].sum().item()
        shares['null'] += null[ids != 0].sum().item()
        n_queries += (ids != 0).sum().item()

    def label(o):
        return f'≤{o}' if o < -MAX_OFFSET else f'≥{o}' if o > MAX_OFFSET else str(o)

    per_offset = pd.DataFrame(
        {('same word' if is_same else 'other word'): {label(o): sums[o, is_same] / counts[o, is_same]
                                                       if counts[o, is_same] else np.nan
                                                       for o in range(-MAX_OFFSET - 1, MAX_OFFSET + 2)}
         for is_same in (True, False)})
    return per_offset, {key: value / n_queries for key, value in shares.items()}


profile_names = [name for name in cloze_models if name.endswith(f' {PROFILE_VARIANT}')]
profile_seqs = cloze_eval_seqs['zap']
profiles = {name: attention_profile(cloze_models[name], profile_seqs) for name in profile_names}

display(pd.DataFrame({name: shares for name, (_, shares) in profiles.items()}).T.round(3))
fig, axes = plt.subplots(1, len(profiles), figsize=(5 * len(profiles), 3.2), sharey=True, squeeze=False)
for ax, (name, (per_offset, _)) in zip(axes[0], profiles.items()):
    per_offset.plot(ax=ax, marker='o')
    ax.set_title(name)
    ax.set_xticks(range(len(per_offset)))
    ax.set_xticklabels(per_offset.index, rotation=90)
    ax.set_xlabel('offset of the attended character')
axes[0, 0].set_ylabel('mean attention per character')
plt.tight_layout()
plt.show()
for name, (per_offset, _) in profiles.items():
    print(name)
    display(per_offset.T.round(4))

### Attention maps for one word and one line

Rows are the predicted positions, columns the attended characters (`∅` is the null slot), averaged over heads and layers. The diagonal is always empty: a position never sees its own character. `EXAMPLE_TEXTS` can hold any text; characters outside the CTC set are dropped.

In [ ]:
example_line = next((t for t in lm_eval_texts['zap'] if 25 <= len(t) <= 45 and ' ' in t), lm_eval_texts['zap'][0])
EXAMPLE_TEXTS = ['dzizea', example_line]


@torch.no_grad()
def attention_map(cloze_lm, text):
    ids = pad_ids(cloze_sequences([text])).to(DEVICE)
    _, weights = cloze_lm.eval()(ids, return_weights=True)
    return torch.stack(weights).float().mean((0, 2))[0].cpu().numpy()


for text in EXAMPLE_TEXTS:
    chars = [ctc_chars[i - 1] for i in cloze_sequences([text])[0]]
    shown = [c if c != ' ' else '␣' for c in chars]
    fig, axes = plt.subplots(1, len(profile_names), figsize=(0.32 * len(chars) * len(profile_names) + 2,
                                                             0.3 * len(chars) + 1.5), squeeze=False)
    for ax, name in zip(axes[0], profile_names):
        ax.imshow(attention_map(cloze_models[name], text), cmap='viridis')
        ax.set_xticks(range(len(chars) + 1))
        ax.set_xticklabels(['∅'] + shown)
        ax.set_yticks(range(len(chars)))
        ax.set_yticklabels(shown)
        ax.set_title(name)
    plt.tight_layout()
    plt.show()

## Correct the CTC reading and choose β on validation

As in Part 5: each emitted character's CTC distribution and the cloze distribution are fused as `log p_ctc + β · log p_cloze`, twice. β = 0 is the CTC reading alone. For each model, β is chosen on validation; the local configuration to report is the one with the lowest validation CER.

In [ ]:
@torch.no_grad()
def ctc_readings(split):
    """Greedy CTC reading of each line, as each emitted character's log-distribution over classes (blank removed)."""
    loader = DataLoader(split, batch_size=CTC_BATCH_SIZE, collate_fn=ctc_collator, num_workers=NUM_WORKERS)
    ctc_model.eval()
    readings = []
    for batch in loader:
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            logits = ctc_model(batch['pixel_values'].to(DEVICE))
        for probs in logits.float().softmax(-1).cpu():
            slots, prev = [], 0
            for t, i in enumerate(probs.argmax(-1).tolist()):
                if i != 0 and i != prev:
                    slots.append(t)
                elif i != 0 and probs[t, i] > probs[slots[-1], i]:
                    slots[-1] = t
                prev = i
            char_probs = probs[slots].clone()
            char_probs[:, 0] = 0.0
            char_probs = char_probs / char_probs.sum(-1, keepdim=True).clamp_min(1e-8)
            readings.append(char_probs.clamp_min(1e-8).log())
    return readings


@torch.no_grad()
def cloze_correct(cloze_lm, readings, beta):
    lines = []
    for start in range(0, len(readings), CLOZE_BATCH_SIZE):
        chunk = readings[start:start + CLOZE_BATCH_SIZE]
        n = max(max(r.size(0) for r in chunk), 1)
        ctc_logp = torch.zeros(len(chunk), n, n_ctc_classes)
        for row, reading in enumerate(chunk):
            ctc_logp[row, :reading.size(0)] = reading
        ctc_logp = ctc_logp.to(DEVICE)
        lengths = torch.tensor([r.size(0) for r in chunk], device=DEVICE)
        real = torch.arange(n, device=DEVICE)[None, :] < lengths[:, None]
        ids = torch.where(real, ctc_logp.argmax(-1), 0)
        for _ in range(CLOZE_ITERATIONS if beta else 0):
            fused = ctc_logp + beta * cloze_lm(ids).float().log_softmax(-1)
            fused[..., 0] = float('-inf')
            ids = torch.where(real, fused.argmax(-1), 0)
        for row in ids.tolist():
            text = ''.join(ctc_chars[i - 1] for i in row if i)
            lines.append(re.sub(r'\s+', ' ', text).strip())
    return lines


val_readings, test_readings = ctc_readings(ds['validation']), ctc_readings(ds['test'])
val_references = [str(t or '').strip() for t in ds['validation']['text']]
correction_results, best_betas, test_predictions = {}, {}, {}
for name, cloze_lm in cloze_models.items():
    val_cer = {beta: cer(val_references, cloze_correct(cloze_lm, val_readings, beta)) for beta in CLOZE_BETAS}
    best_betas[name] = min(val_cer, key=val_cer.get)
    correction_results[name] = {f'beta={beta}': c for beta, c in val_cer.items()}
    test_predictions[f'CTC + cloze {name}'] = cloze_correct(cloze_lm, test_readings, best_betas[name])
    print(f'{name}: best beta on validation = {best_betas[name]}')

# Validation CER for each beta; beta=0 is the CTC reading alone.
validation_table = pd.DataFrame(correction_results).T
validation_table['best'] = validation_table.min(1)
display(validation_table.round(4))
for variant in CLOZE_VARIANTS:
    local = validation_table.loc[[n for n in validation_table.index if n.startswith('local') and n.endswith(f' {variant}')], 'best']
    print(f'{variant}: best local configuration on validation = {local.idxmin()}')
json.dump({'best_betas': best_betas, 'validation_cer': correction_results, 'configs': LOCAL_CONFIGS},
          open(os.path.join(LOCAL_DIR, 'validation_results.json'), 'w'), indent=1)

## Test evaluation

The CTC reading alone, the original cloze correction and the local ones, next to the test predictions the modified-transformer notebook saved (pretrained baseline, bridges, character decoders with and without reranking), when its test evaluation has run. Scores are broken down by language, and repeated without Za687 for comparison with `ticha_trocr_evaluation.ipynb`.

In [ ]:
references = [str(t or '').strip() for t in ds['test']['text']]


def prediction_frame(predictions):
    return pd.DataFrame({
        'id': ds['test']['id'],
        'doc_id': ds['test']['doc_id'],
        'reference': references,
        'prediction': [p.strip() for p in predictions],
        'language': ds['test']['language']
    })


def scores_by_language(name, df):
    rows = []
    for language, group in [('all', df), *df.groupby('language')]:
        rows.append({'model': name, 'language': language, 'lines': len(group),
                     'cer': cer(list(group['reference']), list(group['prediction'])),
                     'wer': wer(list(group['reference']), list(group['prediction']))})
    return rows


test_frames = {'encoder CTC only': prediction_frame(ctc_predict(ctc_model, ds['test'])[0])}
test_frames.update({name: prediction_frame(predictions) for name, predictions in test_predictions.items()})
for name, df in test_frames.items():
    df.to_csv(os.path.join(LOCAL_DIR, f"test_predictions_{name.replace(' ', '_')}.csv"), index=False)

# The modified-transformer notebook's saved predictions, if its test evaluation has run.
for path in sorted(p for p in os.listdir(OUTPUT_DIR) if p.startswith('test_predictions_') and p.endswith('.csv')):
    name = path[len('test_predictions_'):-len('.csv')].replace('_', ' ')
    if name in test_frames or name.startswith('CTC + cloze'):  # already here as 'CTC + cloze original ...'
        continue
    df = pd.read_csv(os.path.join(OUTPUT_DIR, path), keep_default_na=False)
    df['reference'], df['prediction'] = df['reference'].astype(str), df['prediction'].astype(str)
    if list(df['reference']) != references:
        print(f'Skipping {path}: its test lines differ from this notebook\'s')
        continue
    test_frames[name] = df

scores = pd.DataFrame([row for name, df in test_frames.items() for row in scores_by_language(name, df)])
scores.to_csv(os.path.join(LOCAL_DIR, 'test_scores_by_language.csv'), index=False)
display(scores.pivot(index='model', columns='language', values='cer').round(3))

COMPARABLE_EXCLUDE = {'Za687'}
comparable = pd.DataFrame([row for name, df in test_frames.items()
                           for row in scores_by_language(name, df[~df['doc_id'].isin(COMPARABLE_EXCLUDE)])])
comparable.to_csv(os.path.join(LOCAL_DIR, 'test_scores_by_language_without_Za687.csv'), index=False)
print('Without Za687 (comparable with ticha_trocr_evaluation.ipynb):')
display(comparable.pivot(index='model', columns='language', values='cer').round(3))

## How to report results

- The question: does restricting the cloze model's attention to nearby letters, with other words visible but penalized, correct more of the CTC encoder's errors than the unrestricted model? Compare `CTC + cloze local … ` with `CTC + cloze original …` and with `encoder CTC only`, on the Zapotec column first.
- Report the configuration (`W`, `γ`, the per-head slopes 2^−(h+1)) and β for each model. All of them were chosen on validation; the test split is evaluated once.
- Use the attention profile to describe what the original model learned on its own: whether it already favoured neighbouring letters and the current word, and how much attention it gave to other words.
- Limits: the correction fixes substituted characters only (the length comes from CTC), and word boundaries come from the CTC reading, so a misread space shifts them.